In [1]:
# CELL 1: IMPORTS & SETUP

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os, joblib, time
from itertools import combinations
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, recall_score
from imblearn.over_sampling import SMOTE
import xgboost as xgb

os.makedirs('../results', exist_ok=True)
sns.set_style('whitegrid')
plt.rcParams.update({'figure.figsize': (13, 5), 'font.size': 11})

print('=' * 60)
print('    NOTEBOOK 04: ADVANCED FEATURE ENGINEERING')
print('=' * 60)
print('Goal: Build better features → improve XGBoost fraud detection')

    NOTEBOOK 04: ADVANCED FEATURE ENGINEERING
Goal: Build better features → improve XGBoost fraud detection


In [2]:
# CELL 2: LOAD RAW DATA

try:
    df = pd.read_csv('../data/creditcard.csv')
    print(' Full dataset loaded')
except FileNotFoundError:
    df = pd.read_csv('../data/sample_creditcard.csv')
    print('  Using sample dataset')

# Also load what NB02 saved (so we know the baseline features)
nb02_features = pd.read_csv('../data/feature_names.csv')['feature'].tolist()

print(f'Dataset shape : {df.shape}')
print(f'Fraud rate    : {df["Class"].mean()*100:.4f}%')
print(f'NB02 features : {len(nb02_features)} → {nb02_features}')

 Full dataset loaded
Dataset shape : (284807, 31)
Fraud rate    : 0.1727%
NB02 features : 32 → ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Hour_sin', 'Hour_cos', 'Log_Amount']


In [3]:
# CELL 3: REPRODUCE NB02 FEATURES (baseline)

print('=== Reproducing NB02 baseline features ===')

df_base = df.copy()

# Exactly as NB02:
df_base['Hour']      = (df_base['Time'] / 3600) % 24
df_base['Hour_sin']  = np.sin(2 * np.pi * df_base['Hour'] / 24)
df_base['Hour_cos']  = np.cos(2 * np.pi * df_base['Hour'] / 24)
df_base['Log_Amount'] = np.log1p(df_base['Amount'])
df_base = df_base.drop(columns=['Time', 'Hour'])

baseline_features = [c for c in df_base.columns if c != 'Class']
print(f'Baseline feature count: {len(baseline_features)}')
print(f'Features: {baseline_features}')

=== Reproducing NB02 baseline features ===
Baseline feature count: 32
Features: ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Hour_sin', 'Hour_cos', 'Log_Amount']


In [4]:
# CELL 4: INTERACTION FEATURES
# XGBoost already finds interactions, but providing them
# explicitly can help the model find them faster and
# with lower depth trees (less overfitting risk)

print('=== A. Interaction Features (top SHAP pairs) ===')
print('Rationale: NB03 SHAP showed V14, V12, V10, V4 are top predictors')
print('  Interaction = product of two features → captures joint effect')

df_eng = df_base.copy()

# Top 6 features identified in NB03 SHAP analysis
top_shap_features = ['V14', 'V12', 'V10', 'V4', 'V11', 'V17']

# All pairwise products of top 4 → 6 interaction terms
interaction_cols = []
for f1, f2 in combinations(top_shap_features[:4], 2):
    col_name = f'{f1}_x_{f2}'
    df_eng[col_name] = df_eng[f1] * df_eng[f2]
    interaction_cols.append(col_name)

print(f'\nCreated {len(interaction_cols)} interaction features:')
for col in interaction_cols:
    f_fraud = df_eng[df_eng['Class']==1][col].mean()
    f_legit = df_eng[df_eng['Class']==0][col].mean()
    sep_ratio = abs(f_fraud - f_legit) / (abs(f_legit) + 1e-8)
    print(f'  {col:15s} | fraud_mean={f_fraud:7.3f} | legit_mean={f_legit:7.3f} | sep={sep_ratio:.3f}')

=== A. Interaction Features (top SHAP pairs) ===
Rationale: NB03 SHAP showed V14, V12, V10, V4 are top predictors
  Interaction = product of two features → captures joint effect

Created 6 interaction features:
  V14_x_V12       | fraud_mean= 59.536 | legit_mean= -0.103 | sep=578.876
  V14_x_V10       | fraud_mean= 51.790 | legit_mean= -0.090 | sep=578.876
  V14_x_V4        | fraud_mean=-39.557 | legit_mean=  0.068 | sep=578.876
  V12_x_V10       | fraud_mean= 54.513 | legit_mean= -0.094 | sep=578.876
  V12_x_V4        | fraud_mean=-38.824 | legit_mean=  0.067 | sep=578.876
  V10_x_V4        | fraud_mean=-35.993 | legit_mean=  0.062 | sep=578.876
